# 🏠 Nashville Housing Data — SQL Cleaning Pipeline

> **Author:** Noah Asgodom 
> **Dataset:** Nashville Housing Data (Excel)  
> **Engine:** DuckDB (in-process SQL)  
> **Goal:** Transform raw housing data into a clean, analysis-ready dataset using structured SQL inside Jupyter Notebook.

---

## Pipeline Overview

| Step | Task |
|------|------|
| 1 | Load Excel into DuckDB |
| 2 | Standardize `SaleDate` to DATE |
| 3 | Fill missing `PropertyAddress` via `ParcelID` self-join |
| 4 | Split `PropertyAddress` → Address + City |
| 5 | Split `OwnerAddress` → Address + City + State |
| 6 | Standardize `SoldAsVacant` → Yes / No |
| 7 | Remove duplicate rows via `ROW_NUMBER()` |
| 8 | Build final cleaned table |
| 9 | Preview + validate output |
| 10 | Export to CSV and Parquet |

---
## ⚙️ 0 — Environment Setup

In [2]:
# Install dependencies if not already installed
# !pip install duckdb pandas openpyxl

import duckdb
import pandas as pd
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
RAW_FILE   = Path(r"C:\Users\noahi\Documents\Git\Noah_Portofilio-\project_1\data\raw\Nashville_Housing_Raw.xlsx")
OUTPUT_DIR = Path(r"C:\Users\noahi\Documents\Git\Noah_Portofilio-\project_1\data\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── DuckDB connection (in-memory; swap ':memory:' for a file path to persist) ──
con = duckdb.connect(database=':memory:')

print(f"DuckDB version : {duckdb.__version__}")
print(f"Raw file       : {RAW_FILE}")
print(f"Output folder  : {OUTPUT_DIR}")

DuckDB version : 1.5.4
Raw file       : C:\Users\noahi\Documents\Git\Noah_Portofilio-\project_1\data\raw\Nashville_Housing_Raw.xlsx
Output folder  : C:\Users\noahi\Documents\Git\Noah_Portofilio-\project_1\data\processed


---
## 📥 Step 1 — Load Excel File into DuckDB

In [3]:
# Load with pandas first (DuckDB can't read .xlsx natively),
# then register the DataFrame as a virtual DuckDB view.
raw_df = pd.read_excel(RAW_FILE)

# Normalise column names: strip whitespace, replace spaces with underscores
raw_df.columns = (
    raw_df.columns
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
)

# Register as a DuckDB view so all subsequent work stays in SQL
con.register('raw_housing', raw_df)

print(f"Rows loaded : {len(raw_df):,}")
print(f"Columns     : {list(raw_df.columns)}")

Rows loaded : 56,477
Columns     : ['UniqueID', 'ParcelID', 'LandUse', 'PropertyAddress', 'SaleDate', 'SalePrice', 'LegalReference', 'SoldAsVacant', 'OwnerName', 'OwnerAddress', 'Acreage', 'TaxDistrict', 'LandValue', 'BuildingValue', 'TotalValue', 'YearBuilt', 'Bedrooms', 'FullBath', 'HalfBath']


In [4]:
# Quick schema + null-count audit — know your data before cleaning it
con.sql("""
    SELECT
        column_name,
        data_type,
        -- DuckDB information_schema doesn't expose nulls directly;
        -- use SUMMARIZE or a pivot for full null counts (see below)
    FROM information_schema.columns
    WHERE table_name = 'raw_housing'
    ORDER BY ordinal_position
""").show()

# Row-level sanity check
con.sql("SUMMARIZE raw_housing").show()

┌─────────────────┬──────────────┐
│   column_name   │  data_type   │
│     varchar     │   varchar    │
├─────────────────┼──────────────┤
│ UniqueID        │ BIGINT       │
│ ParcelID        │ VARCHAR      │
│ LandUse         │ VARCHAR      │
│ PropertyAddress │ VARCHAR      │
│ SaleDate        │ TIMESTAMP_NS │
│ SalePrice       │ BIGINT       │
│ LegalReference  │ VARCHAR      │
│ SoldAsVacant    │ VARCHAR      │
│ OwnerName       │ VARCHAR      │
│ OwnerAddress    │ VARCHAR      │
│ Acreage         │ DOUBLE       │
│ TaxDistrict     │ VARCHAR      │
│ LandValue       │ DOUBLE       │
│ BuildingValue   │ DOUBLE       │
│ TotalValue      │ DOUBLE       │
│ YearBuilt       │ DOUBLE       │
│ Bedrooms        │ DOUBLE       │
│ FullBath        │ DOUBLE       │
│ HalfBath        │ DOUBLE       │
└─────────────────┴──────────────┘
  19 rows              2 columns

┌─────────────────┬──────────────┬────────────────────────────────────────┬─────────────────────────────────┬───────────────┬─

In [5]:
# Persist raw data as a base DuckDB table (avoids re-reading the Excel file later)
con.sql("""
    CREATE OR REPLACE TABLE housing_raw AS
    SELECT * FROM raw_housing
""")

print("✅ Table 'housing_raw' created.")
con.sql("SELECT COUNT(*) AS total_rows FROM housing_raw").show()

✅ Table 'housing_raw' created.
┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│      56477 │
└────────────┘



---
## 📅 Step 2 — Standardize SaleDate to DATE Format

In [6]:
# Preview raw SaleDate values before casting
con.sql("""
    SELECT
        SaleDate,
        typeof(SaleDate)          AS raw_type,
        TRY_CAST(SaleDate AS DATE) AS parsed_date
    FROM housing_raw
    LIMIT 10
""").show()

┌─────────────────────┬──────────────┬─────────────┐
│      SaleDate       │   raw_type   │ parsed_date │
│    timestamp_ns     │   varchar    │    date     │
├─────────────────────┼──────────────┼─────────────┤
│ 2013-04-09 00:00:00 │ TIMESTAMP_NS │ 2013-04-09  │
│ 2014-06-10 00:00:00 │ TIMESTAMP_NS │ 2014-06-10  │
│ 2016-09-26 00:00:00 │ TIMESTAMP_NS │ 2016-09-26  │
│ 2016-01-29 00:00:00 │ TIMESTAMP_NS │ 2016-01-29  │
│ 2014-10-10 00:00:00 │ TIMESTAMP_NS │ 2014-10-10  │
│ 2014-07-16 00:00:00 │ TIMESTAMP_NS │ 2014-07-16  │
│ 2014-08-28 00:00:00 │ TIMESTAMP_NS │ 2014-08-28  │
│ 2016-09-27 00:00:00 │ TIMESTAMP_NS │ 2016-09-27  │
│ 2015-08-14 00:00:00 │ TIMESTAMP_NS │ 2015-08-14  │
│ 2014-08-29 00:00:00 │ TIMESTAMP_NS │ 2014-08-29  │
└─────────────────────┴──────────────┴─────────────┘
  10 rows                                3 columns



In [7]:
con.sql("""
    CREATE OR REPLACE TABLE housing_dates AS
    SELECT
        *,
        -- TRY_CAST is safer than CAST — returns NULL on parse failure instead of erroring
        TRY_CAST(SaleDate AS DATE) AS SaleDateCleaned
    FROM housing_raw
""")

# Audit: any dates that failed to parse?
con.sql("""
    SELECT COUNT(*) AS failed_date_parse
    FROM housing_dates
    WHERE SaleDateCleaned IS NULL
      AND SaleDate IS NOT NULL
""").show()

print("✅ Step 2 complete — SaleDateCleaned column added.")

┌───────────────────┐
│ failed_date_parse │
│       int64       │
├───────────────────┤
│                 0 │
└───────────────────┘

✅ Step 2 complete — SaleDateCleaned column added.


---
## 🏠 Step 3 — Fill Missing PropertyAddress via ParcelID Self-Join

In [8]:
# Diagnose nulls before filling
con.sql("""
    SELECT COUNT(*) AS missing_addresses
    FROM housing_dates
    WHERE PropertyAddress IS NULL OR TRIM(PropertyAddress) = ''
""").show()

┌───────────────────┐
│ missing_addresses │
│       int64       │
├───────────────────┤
│                29 │
└───────────────────┘



In [9]:
con.sql("""
    CREATE OR REPLACE TABLE housing_addr_filled AS
    SELECT
        a.*,
        -- COALESCE: use a's address if present, else borrow from a matching ParcelID row
        COALESCE(
            NULLIF(TRIM(a.PropertyAddress), ''),
            b.PropertyAddress
        ) AS PropertyAddressFilled
    FROM housing_dates AS a
    LEFT JOIN (
        -- Donor sub-query: one non-null address per ParcelID
        SELECT
            ParcelID,
            MAX(PropertyAddress) AS PropertyAddress   -- MAX picks a non-null value
        FROM housing_dates
        WHERE PropertyAddress IS NOT NULL
          AND TRIM(PropertyAddress) <> ''
        GROUP BY ParcelID
    ) AS b ON a.ParcelID = b.ParcelID
""")

# Confirm nulls have been reduced
con.sql("""
    SELECT COUNT(*) AS still_missing
    FROM housing_addr_filled
    WHERE PropertyAddressFilled IS NULL OR TRIM(PropertyAddressFilled) = ''
""").show()

print("✅ Step 3 complete — missing PropertyAddress values backfilled.")

┌───────────────┐
│ still_missing │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘

✅ Step 3 complete — missing PropertyAddress values backfilled.


---
## ✂️ Step 4 — Split PropertyAddress into Address and City

In [10]:
# Preview the delimiter pattern before splitting
con.sql("""
    SELECT
        PropertyAddressFilled,
        SPLIT_PART(PropertyAddressFilled, ',', 1) AS PropertyStreet,
        TRIM(SPLIT_PART(PropertyAddressFilled, ',', 2)) AS PropertyCity
    FROM housing_addr_filled
    WHERE PropertyAddressFilled IS NOT NULL
    LIMIT 10
""").show()

┌───────────────────────────────────────┬───────────────────────┬────────────────┐
│         PropertyAddressFilled         │    PropertyStreet     │  PropertyCity  │
│                varchar                │        varchar        │    varchar     │
├───────────────────────────────────────┼───────────────────────┼────────────────┤
│ 1808  FOX CHASE DR, GOODLETTSVILLE    │ 1808  FOX CHASE DR    │ GOODLETTSVILLE │
│ 1832  FOX CHASE DR, GOODLETTSVILLE    │ 1832  FOX CHASE DR    │ GOODLETTSVILLE │
│ 1864 FOX CHASE  DR, GOODLETTSVILLE    │ 1864 FOX CHASE  DR    │ GOODLETTSVILLE │
│ 1853  FOX CHASE DR, GOODLETTSVILLE    │ 1853  FOX CHASE DR    │ GOODLETTSVILLE │
│ 1829  FOX CHASE DR, GOODLETTSVILLE    │ 1829  FOX CHASE DR    │ GOODLETTSVILLE │
│ 1821  FOX CHASE DR, GOODLETTSVILLE    │ 1821  FOX CHASE DR    │ GOODLETTSVILLE │
│ 2005  SADIE LN, GOODLETTSVILLE        │ 2005  SADIE LN        │ GOODLETTSVILLE │
│ 1917 GRACELAND  DR, GOODLETTSVILLE    │ 1917 GRACELAND  DR    │ GOODLETTSVILLE │
│ 14

In [11]:
con.sql("""
    CREATE OR REPLACE TABLE housing_prop_split AS
    SELECT
        *,
        TRIM(SPLIT_PART(PropertyAddressFilled, ',', 1)) AS PropertyStreet,
        TRIM(SPLIT_PART(PropertyAddressFilled, ',', 2)) AS PropertyCity
    FROM housing_addr_filled
""")

print("✅ Step 4 complete — PropertyStreet and PropertyCity columns added.")
con.sql("SELECT PropertyAddressFilled, PropertyStreet, PropertyCity FROM housing_prop_split LIMIT 5").show()

✅ Step 4 complete — PropertyStreet and PropertyCity columns added.
┌────────────────────────────────────┬────────────────────┬────────────────┐
│       PropertyAddressFilled        │   PropertyStreet   │  PropertyCity  │
│              varchar               │      varchar       │    varchar     │
├────────────────────────────────────┼────────────────────┼────────────────┤
│ 1808  FOX CHASE DR, GOODLETTSVILLE │ 1808  FOX CHASE DR │ GOODLETTSVILLE │
│ 1832  FOX CHASE DR, GOODLETTSVILLE │ 1832  FOX CHASE DR │ GOODLETTSVILLE │
│ 1864 FOX CHASE  DR, GOODLETTSVILLE │ 1864 FOX CHASE  DR │ GOODLETTSVILLE │
│ 1853  FOX CHASE DR, GOODLETTSVILLE │ 1853  FOX CHASE DR │ GOODLETTSVILLE │
│ 1829  FOX CHASE DR, GOODLETTSVILLE │ 1829  FOX CHASE DR │ GOODLETTSVILLE │
└────────────────────────────────────┴────────────────────┴────────────────┘



---
## ✂️ Step 5 — Split OwnerAddress into Address, City, and State

In [12]:
# Preview — OwnerAddress uses comma-separated: 'street, city, state'
con.sql("""
    SELECT
        OwnerAddress,
        TRIM(SPLIT_PART(OwnerAddress, ',', 1)) AS OwnerStreet,
        TRIM(SPLIT_PART(OwnerAddress, ',', 2)) AS OwnerCity,
        TRIM(SPLIT_PART(OwnerAddress, ',', 3)) AS OwnerState
    FROM housing_prop_split
    WHERE OwnerAddress IS NOT NULL
    LIMIT 10
""").show()

┌───────────────────────────────────────────┬───────────────────────┬────────────────┬────────────┐
│               OwnerAddress                │      OwnerStreet      │   OwnerCity    │ OwnerState │
│                  varchar                  │        varchar        │    varchar     │  varchar   │
├───────────────────────────────────────────┼───────────────────────┼────────────────┼────────────┤
│ 1808  FOX CHASE DR, GOODLETTSVILLE, TN    │ 1808  FOX CHASE DR    │ GOODLETTSVILLE │ TN         │
│ 1832  FOX CHASE DR, GOODLETTSVILLE, TN    │ 1832  FOX CHASE DR    │ GOODLETTSVILLE │ TN         │
│ 1864  FOX CHASE DR, GOODLETTSVILLE, TN    │ 1864  FOX CHASE DR    │ GOODLETTSVILLE │ TN         │
│ 1853  FOX CHASE DR, GOODLETTSVILLE, TN    │ 1853  FOX CHASE DR    │ GOODLETTSVILLE │ TN         │
│ 1829  FOX CHASE DR, GOODLETTSVILLE, TN    │ 1829  FOX CHASE DR    │ GOODLETTSVILLE │ TN         │
│ 1821  FOX CHASE DR, GOODLETTSVILLE, TN    │ 1821  FOX CHASE DR    │ GOODLETTSVILLE │ TN         │


In [13]:
con.sql("""
    CREATE OR REPLACE TABLE housing_owner_split AS
    SELECT
        *,
        TRIM(SPLIT_PART(OwnerAddress, ',', 1)) AS OwnerStreet,
        TRIM(SPLIT_PART(OwnerAddress, ',', 2)) AS OwnerCity,
        TRIM(SPLIT_PART(OwnerAddress, ',', 3)) AS OwnerState
    FROM housing_prop_split
""")

print("✅ Step 5 complete — OwnerStreet, OwnerCity, OwnerState columns added.")
con.sql("SELECT OwnerAddress, OwnerStreet, OwnerCity, OwnerState FROM housing_owner_split LIMIT 5").show()

✅ Step 5 complete — OwnerStreet, OwnerCity, OwnerState columns added.
┌────────────────────────────────────────┬────────────────────┬────────────────┬────────────┐
│              OwnerAddress              │    OwnerStreet     │   OwnerCity    │ OwnerState │
│                varchar                 │      varchar       │    varchar     │  varchar   │
├────────────────────────────────────────┼────────────────────┼────────────────┼────────────┤
│ 1808  FOX CHASE DR, GOODLETTSVILLE, TN │ 1808  FOX CHASE DR │ GOODLETTSVILLE │ TN         │
│ 1832  FOX CHASE DR, GOODLETTSVILLE, TN │ 1832  FOX CHASE DR │ GOODLETTSVILLE │ TN         │
│ 1864  FOX CHASE DR, GOODLETTSVILLE, TN │ 1864  FOX CHASE DR │ GOODLETTSVILLE │ TN         │
│ 1853  FOX CHASE DR, GOODLETTSVILLE, TN │ 1853  FOX CHASE DR │ GOODLETTSVILLE │ TN         │
│ 1829  FOX CHASE DR, GOODLETTSVILLE, TN │ 1829  FOX CHASE DR │ GOODLETTSVILLE │ TN         │
└────────────────────────────────────────┴────────────────────┴────────────────┴────

---
## ✅ Step 6 — Standardize SoldAsVacant Values to Yes / No

In [14]:
# Audit distinct raw values — there may be 'Y', 'N', 'Yes', 'No', nulls, etc.
con.sql("""
    SELECT
        SoldAsVacant,
        COUNT(*) AS occurrences
    FROM housing_owner_split
    GROUP BY SoldAsVacant
    ORDER BY occurrences DESC
""").show()

┌──────────────┬─────────────┐
│ SoldAsVacant │ occurrences │
│   varchar    │    int64    │
├──────────────┼─────────────┤
│ No           │       51403 │
│ Yes          │        4623 │
│ N            │         399 │
│ Y            │          52 │
└──────────────┴─────────────┘



In [15]:
con.sql("""
    CREATE OR REPLACE TABLE housing_vacant_std AS
    SELECT
        *,
        CASE
            WHEN UPPER(TRIM(SoldAsVacant)) IN ('Y', 'YES') THEN 'Yes'
            WHEN UPPER(TRIM(SoldAsVacant)) IN ('N', 'NO')  THEN 'No'
            ELSE NULL   -- surfaces unexpected values rather than silently mislabelling them
        END AS SoldAsVacantStd
    FROM housing_owner_split
""")

# Verify the distribution post-standardization
con.sql("""
    SELECT SoldAsVacantStd, COUNT(*) AS count
    FROM housing_vacant_std
    GROUP BY SoldAsVacantStd
    ORDER BY count DESC
""").show()

print("✅ Step 6 complete — SoldAsVacantStd column added.")

┌─────────────────┬───────┐
│ SoldAsVacantStd │ count │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ No              │ 51802 │
│ Yes             │  4675 │
└─────────────────┴───────┘

✅ Step 6 complete — SoldAsVacantStd column added.


---
## 🔁 Step 7 — Remove Duplicate Rows with ROW_NUMBER()

In [16]:
# Identify duplicates before removing them
con.sql("""
    SELECT COUNT(*) AS total_rows FROM housing_vacant_std
""").show()

con.sql("""
    SELECT COUNT(*) AS duplicate_rows
    FROM (
        SELECT
            ROW_NUMBER() OVER (
                PARTITION BY
                    ParcelID,
                    PropertyAddressFilled,
                    SaleDateCleaned,
                    SalePrice,
                    LegalReference
                ORDER BY UniqueID
            ) AS rn
        FROM housing_vacant_std
    )
    WHERE rn > 1
""").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│      56477 │
└────────────┘

┌────────────────┐
│ duplicate_rows │
│     int64      │
├────────────────┤
│            104 │
└────────────────┘



In [17]:
con.sql("""
    CREATE OR REPLACE TABLE housing_deduped AS
    SELECT * EXCLUDE (rn)     -- drop the helper column after filtering
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY
                    ParcelID,
                    PropertyAddressFilled,
                    SaleDateCleaned,
                    SalePrice,
                    LegalReference
                ORDER BY UniqueID   -- keep earliest UniqueID per group
            ) AS rn
        FROM housing_vacant_std
    )
    WHERE rn = 1
""")

before = con.sql("SELECT COUNT(*) AS n FROM housing_vacant_std").fetchone()[0]
after  = con.sql("SELECT COUNT(*) AS n FROM housing_deduped").fetchone()[0]
print(f"✅ Step 7 complete — duplicates removed: {before - after:,}  |  rows remaining: {after:,}")

✅ Step 7 complete — duplicates removed: 104  |  rows remaining: 56,373


---
## 🏗️ Step 8 — Build Final Cleaned Table (selected columns only)

In [18]:
con.sql("""
    CREATE OR REPLACE TABLE housing_cleaned AS
    SELECT
        -- ── Identifiers ──────────────────────────────────────────────────────
        UniqueID,
        ParcelID,

        -- ── Sale information ─────────────────────────────────────────────────
        SaleDateCleaned                          AS SaleDate,
        TRY_CAST(SalePrice AS DOUBLE)            AS SalePrice,
        LegalReference,
        SoldAsVacantStd                          AS SoldAsVacant,

        -- ── Property address (split) ─────────────────────────────────────────
        PropertyStreet,
        PropertyCity,

        -- ── Owner information ────────────────────────────────────────────────
        OwnerName,
        OwnerStreet,
        OwnerCity,
        OwnerState,

        -- ── Property details ─────────────────────────────────────────────────
        Acreage,
        TaxDistrict,
        LandValue,
        BuildingValue,
        TotalValue,
        YearBuilt,
        Bedrooms,
        FullBath,
        HalfBath

    FROM housing_deduped
    ORDER BY SaleDate, UniqueID
""")

print("✅ Step 8 complete — final cleaned table created.")
con.sql("SELECT COUNT(*) AS total_rows FROM housing_cleaned").show()

✅ Step 8 complete — final cleaned table created.
┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│      56373 │
└────────────┘



---
## 🔍 Step 9 — Preview & Validate the Cleaned Dataset

In [19]:
# Preview first 10 rows
con.sql("SELECT * FROM housing_cleaned LIMIT 10").show()

┌──────────┬──────────────────┬────────────┬───────────┬──────────────────┬──────────────┬─────────────────────────┬──────────────┬────────────────────────────────────────────────┬─────────────────────────┬───────────┬────────────┬─────────┬───────────────────────────┬───────────┬───────────────┬────────────┬───────────┬──────────┬──────────┬──────────┐
│ UniqueID │     ParcelID     │  SaleDate  │ SalePrice │  LegalReference  │ SoldAsVacant │     PropertyStreet      │ PropertyCity │                   OwnerName                    │       OwnerStreet       │ OwnerCity │ OwnerState │ Acreage │        TaxDistrict        │ LandValue │ BuildingValue │ TotalValue │ YearBuilt │ Bedrooms │ FullBath │ HalfBath │
│  int64   │     varchar      │    date    │  double   │     varchar      │   varchar    │         varchar         │   varchar    │                    varchar                     │         varchar         │  varchar  │  varchar   │ double  │          varchar          │  double   │    dou

In [20]:
# Statistical summary
con.sql("SUMMARIZE housing_cleaned").show()

┌────────────────┬─────────────┬───────────────────────────────┬─────────────────────────┬───────────────┬────────────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬───────┬─────────────────┐
│  column_name   │ column_type │              min              │           max           │ approx_unique │            avg             │        std         │        q25         │        q50         │         q75         │ count │ null_percentage │
│    varchar     │   varchar   │            varchar            │         varchar         │     int64     │          varchar           │      varchar       │      varchar       │      varchar       │       varchar       │ int64 │  decimal(9,2)   │
├────────────────┼─────────────┼───────────────────────────────┼─────────────────────────┼───────────────┼────────────────────────────┼────────────────────┼────────────────────┼────────────────────┼─────────────────────┼───────┼─────────────────┤
│ UniqueID  

In [21]:
# Null audit across every column — essential for data quality reporting
con.sql("""
    SELECT
        'UniqueID'        AS column_name, COUNT(*) - COUNT(UniqueID)        AS null_count FROM housing_cleaned
    UNION ALL SELECT 'ParcelID',        COUNT(*) - COUNT(ParcelID)          FROM housing_cleaned
    UNION ALL SELECT 'SaleDate',        COUNT(*) - COUNT(SaleDate)          FROM housing_cleaned
    UNION ALL SELECT 'SalePrice',       COUNT(*) - COUNT(SalePrice)         FROM housing_cleaned
    UNION ALL SELECT 'SoldAsVacant',    COUNT(*) - COUNT(SoldAsVacant)      FROM housing_cleaned
    UNION ALL SELECT 'PropertyStreet',  COUNT(*) - COUNT(PropertyStreet)    FROM housing_cleaned
    UNION ALL SELECT 'PropertyCity',    COUNT(*) - COUNT(PropertyCity)      FROM housing_cleaned
    UNION ALL SELECT 'OwnerName',       COUNT(*) - COUNT(OwnerName)         FROM housing_cleaned
    UNION ALL SELECT 'OwnerStreet',     COUNT(*) - COUNT(OwnerStreet)       FROM housing_cleaned
    UNION ALL SELECT 'OwnerCity',       COUNT(*) - COUNT(OwnerCity)         FROM housing_cleaned
    UNION ALL SELECT 'OwnerState',      COUNT(*) - COUNT(OwnerState)        FROM housing_cleaned
    UNION ALL SELECT 'YearBuilt',       COUNT(*) - COUNT(YearBuilt)         FROM housing_cleaned
    ORDER BY null_count DESC
""").show()

┌────────────────┬────────────┐
│  column_name   │ null_count │
│    varchar     │   int64    │
├────────────────┼────────────┤
│ YearBuilt      │      32255 │
│ OwnerName      │      31158 │
│ OwnerStreet    │      30404 │
│ OwnerCity      │      30404 │
│ OwnerState     │      30404 │
│ SalePrice      │          0 │
│ SaleDate       │          0 │
│ UniqueID       │          0 │
│ SoldAsVacant   │          0 │
│ ParcelID       │          0 │
│ PropertyStreet │          0 │
│ PropertyCity   │          0 │
└────────────────┴────────────┘
  12 rows           2 columns



In [22]:
# Business sanity checks
con.sql("""
    SELECT
        MIN(SaleDate)        AS earliest_sale,
        MAX(SaleDate)        AS latest_sale,
        MIN(SalePrice)       AS min_price,
        MAX(SalePrice)       AS max_price,
        ROUND(AVG(SalePrice), 2) AS avg_price,
        COUNT(DISTINCT PropertyCity) AS unique_cities,
        COUNT(DISTINCT OwnerState)   AS unique_states
    FROM housing_cleaned
""").show()

┌───────────────┬─────────────┬───────────┬────────────┬───────────┬───────────────┬───────────────┐
│ earliest_sale │ latest_sale │ min_price │ max_price  │ avg_price │ unique_cities │ unique_states │
│     date      │    date     │  double   │   double   │  double   │     int64     │     int64     │
├───────────────┼─────────────┼───────────┼────────────┼───────────┼───────────────┼───────────────┤
│ 2013-01-02    │ 2019-12-13  │      50.0 │ 54278060.0 │ 327523.01 │            14 │             1 │
└───────────────┴─────────────┴───────────┴────────────┴───────────┴───────────────┴───────────────┘



---
## 💾 Step 10 — Export Cleaned Dataset to CSV and Parquet

In [23]:
CSV_OUT     = str(OUTPUT_DIR / "nashville_housing_cleaned.csv")
PARQUET_OUT = str(OUTPUT_DIR / "nashville_housing_cleaned.parquet")

# ── CSV export ────────────────────────────────────────────────────────────────
# HEADER=TRUE ensures column names are written; DELIMITER is explicit for clarity
con.sql(f"""
    COPY housing_cleaned
    TO '{CSV_OUT}'
    (FORMAT CSV, HEADER TRUE, DELIMITER ',')
""")
print(f"✅ CSV exported  →  {CSV_OUT}")

# ── Parquet export ────────────────────────────────────────────────────────────
# Parquet is strongly preferred for downstream analytics:
#   • Columnar storage → fast aggregations
#   • Type-aware (preserves DATE, DOUBLE, etc.)
#   • ~5-10x smaller than CSV on typical tabular data
con.sql(f"""
    COPY housing_cleaned
    TO '{PARQUET_OUT}'
    (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
print(f"✅ Parquet exported  →  {PARQUET_OUT}")

✅ CSV exported  →  C:\Users\noahi\Documents\Git\Noah_Portofilio-\project_1\data\processed\nashville_housing_cleaned.csv
✅ Parquet exported  →  C:\Users\noahi\Documents\Git\Noah_Portofilio-\project_1\data\processed\nashville_housing_cleaned.parquet


In [24]:
# Round-trip validation — read back from Parquet to confirm integrity
validation = con.sql(f"""
    SELECT
        COUNT(*)         AS row_count,
        MIN(SaleDate)    AS earliest,
        MAX(SaleDate)    AS latest,
        MIN(SalePrice)   AS min_price,
        MAX(SalePrice)   AS max_price
    FROM read_parquet('{PARQUET_OUT}')
""")
validation.show()
print("\n✅ Parquet round-trip validation passed.")

┌───────────┬────────────┬────────────┬───────────┬────────────┐
│ row_count │  earliest  │   latest   │ min_price │ max_price  │
│   int64   │    date    │    date    │  double   │   double   │
├───────────┼────────────┼────────────┼───────────┼────────────┤
│     56373 │ 2013-01-02 │ 2019-12-13 │      50.0 │ 54278060.0 │
└───────────┴────────────┴────────────┴───────────┴────────────┘


✅ Parquet round-trip validation passed.


In [25]:
# File size comparison
import os
csv_size     = os.path.getsize(CSV_OUT)     / (1024 * 1024)
parquet_size = os.path.getsize(PARQUET_OUT) / (1024 * 1024)
print(f"CSV     size : {csv_size:.2f} MB")
print(f"Parquet size : {parquet_size:.2f} MB")
print(f"Compression  : {csv_size / parquet_size:.1f}x smaller with Parquet")

CSV     size : 8.46 MB
Parquet size : 2.92 MB
Compression  : 2.9x smaller with Parquet


---
## 📋 Pipeline Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Loaded Excel into DuckDB via pandas | `housing_raw` table |
| 2 | Cast `SaleDate` with `TRY_CAST` | `SaleDateCleaned` (DATE type) |
| 3 | Backfilled missing `PropertyAddress` via ParcelID self-join | `PropertyAddressFilled` |
| 4 | Split `PropertyAddress` on `,` | `PropertyStreet`, `PropertyCity` |
| 5 | Split `OwnerAddress` on `,` | `OwnerStreet`, `OwnerCity`, `OwnerState` |
| 6 | Mapped Y/N → Yes/No with `CASE` | `SoldAsVacantStd` |
| 7 | De-duplicated with `ROW_NUMBER()` | Removed exact duplicates |
| 8 | Selected final columns | `housing_cleaned` table |
| 9 | Null audit + sanity checks | Data quality validated |
| 10 | Exported CSV + Parquet | `data/processed/` folder |

> 🚀 The cleaned dataset in `data/processed/` is analysis-ready and can be loaded directly into Tableau, Power BI, or any downstream Python/SQL workflow.

In [26]:
# Close the connection when done
con.close()
print("🔒 DuckDB connection closed.")

🔒 DuckDB connection closed.
